# GIADA IV-C3 — interfaccia sinaptica integrata
Quattro sinapsi EMS canoniche, burst eccitatori/inibitori misti, shadow causale online e restart nativo. Il notebook non usa rilasci teacher futuri come input. Task33 è autorizzata soltanto se il report finale supera tutti i gate preregistrati.


In [ ]:
from pathlib import Path
import json,subprocess,sys,tempfile,uuid
from IPython.display import display
WORK=Path(tempfile.mkdtemp(prefix='giada_iv_c3_'))
REPO=WORK/'giada';TEACHER=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','--quiet','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--quiet','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','--quiet','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--quiet','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
try:
 import neuron
except ImportError:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 import neuron
assert neuron.__version__.split('+')[0]=='8.2.7'
print({'revision':REVISION,'neuron':neuron.__version__})


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/('giada_iv_c3_'+REVISION[:7]+'_'+uuid.uuid4().hex[:8])
command=[sys.executable,'-u',str(REPO/'scripts/run_iv_c3_integrated_synaptic_interface.py'),'--repo',str(REPO),'--teacher',str(TEACHER),'--output',str(OUTPUT)]
completed=subprocess.run(command,cwd=REPO,check=False)
result=next((OUTPUT/p for p in ('final_report.json','failure_report.json') if (OUTPUT/p).is_file()),None)
report=json.loads(result.read_text()) if result else {'valid':False,'error':f'worker interrupted (exit={completed.returncode})'}
display({k:report.get(k) for k in ('valid','diagnosis','confirmation_case_count','release_support','negative_controls','iv_c3_passed','task33_authorized','error')})
if report.get('confirmation'):
 rows=list(report['confirmation'].values());display({'all_pass':all(x['passed'] for x in rows),'max_receptor_state_error':max(x['max_receptor_state_error'] for x in rows),'max_current_error_na':max(x['max_current_error_na'] for x in rows),'max_restart_state_error':max(x['max_native_restart_state_error'] for x in rows)})
print({'output':str(OUTPUT),'exit':completed.returncode})
